# **Partie 02 - Assistant RAG (Génération)**

**<u>Objectif :</u>** Intégrer un LLM pour générer des réponses finales à partir du contexte retrouvé (chunks pertinents), afin de compléter l'assistant RAG.

**<u>Contexte</u>**

Dans le notebook précédent (01 - Retrieval), on a construit une base de connaissances vectorielle avec FAISS et des fonctions de recherche. On a maintenant besoin d'un modèle de langage qui, à partir d'une question et des extraits pertinents, génère une réponse naturelle et fidèle aux documents de l'hôtel.

**<u>Choix du modèle :</u>** `Llama 3.2 3B`

On retient Llama 3.2 3B pour cette partie génération. Avec 3 milliards de paramètres, il offre un excellent équilibre entre intelligence et rapidité. Quantifié en Q4_K_M, il tient dans ~3.5 Go de RAM et atteint 80-100 tokens par seconde sur CPU, ce qui permet une latence <10ms pour des réponses courtes. Il supporte bien le français et l'anglais, et sa licence Meta permet une utilisation sans restriction.

**<u>Pourquoi pas d'autres modèles ?</u>**
- Inférence lente avec Qwen 2.5 7B est trop lent sur CPU/GPU (15-25 tokens/s).
- Les API comme GPT-3.5 ajoutent une latence réseau incompressible (100-200ms) et une dépendance à Internet.
- Les modèles <1B manquent d'intelligence pour comprendre des questions complexes sur la documentation.
- Nous visons des réponses factuelles, pas un style spécifique.

**<u>Approche technique :</u>** `llama-cpp-python`

On utilise llama-cpp-python plutôt que Hugging Face Transformers pour deux raisons : ses performances CPU optimisées (C++, instructions SIMD) et le format GGUF qui intègre la quantification (Q4_K_M), réduisant mémoire et temps d'inférence.

**Étapes**

1. Installation de llama-cpp-python.
2. Téléchargement du modèle Llama 3.2 3B au format GGUF.
3. Chargement du modèle avec paramètres optimisés (threads, contexte).
4. Test de génération simple.
5. Intégration avec la fonction de retrieval existante.
6. Test sur les 12 scénarios préparés.
7. Évaluation de la performance et de la qualité.

## **Installation Et Configuration De L'Environnement**

In [1]:
# Installation de llama-cpp-python pour l'inférence CPU
# %pip install llama-cpp-python

In [11]:
import os
import shlex
from pathlib import Path
from llama_cpp import Llama

# Configuration des chemins
PROJET_ROOT = Path.cwd().parent.parent
MODELS_DIR = PROJET_ROOT / "models" / "1-rag"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Les modèles seront sauvegardés dans : {MODELS_DIR}")

Les modèles seront sauvegardés dans : /home/kevin/Documents/Collège LaCité/Cours IA en Informatique (51847)/Hiver 2026/Intelligence Artificielle Générative (IFM31124-0020-H2026)/Projets/ua1/assistant-rag/models/1-rag


### **Rechargement Des Données Du Retrieval**

In [19]:
# Import des librairies nécessaires
import pandas as pd
import pickle
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Configuration des chemins
PROJET_ROOT = Path.cwd().parent.parent
DATA_PROCESSED = PROJET_ROOT / "data" / "processed"
MODELS_DIR = PROJET_ROOT / "models" / "1-rag"

print("Chemins configurés.")

Chemins configurés.


In [20]:
# Chargement des chunks
df_chunks = pd.read_parquet(DATA_PROCESSED / "chunks.parquet")
print(f"Chunks chargés : {len(df_chunks)}")
df_chunks.head()

Chunks chargés : 148


,source,chunk_id,texte,start_char,end_char
0,Principes de vie internes.pdf,Principes de vie internes.pdf_0,"PRINCIPES DE VIE INTERNES\nÀ notre équipe,\nAu...",0,512
1,Principes de vie internes.pdf,Principes de vie internes.pdf_462,"ptionnelle pour notre clientèle, de développer...",462,974
2,Principes de vie internes.pdf,Principes de vie internes.pdf_924,it d’équipe\nNous travaillons en collaboration...,924,1436
3,Principes de vie internes.pdf,Principes de vie internes.pdf_1386,"d'éthique et d'intégrité, en\nfaisant preuve d...",1386,1898
4,Principes de vie internes.pdf,Principes de vie internes.pdf_1848,action. Nous portons attention à leurs besoins...,1848,2360


In [21]:
# Chargement de l'index FAISS
index = faiss.read_index(str(MODELS_DIR / "index.faiss"))
print(f"Index FAISS chargé avec {index.ntotal} vecteurs")

Index FAISS chargé avec 148 vecteurs


In [22]:
# Chargement du modèle d'embedding
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Modèle d'embedding chargé.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modèle d'embedding chargé.


In [23]:
# Définition de la fonction de recherche
def rechercher_contexte(question, k=3):
  question_embedding = embedding_model.encode([question])
  distances, indices = index.search(question_embedding.astype('float32'), k)
  
  contexte = ""
  sources = []
  
  for i, idx in enumerate(indices[0]):
    chunk = df_chunks.iloc[idx]
    contexte += f"[Extrait {i+1} - {chunk['source']}]\n{chunk['texte']}\n\n"
    sources.append(chunk['source'])
  
  return contexte, sources

In [24]:
# Test rapide
test_question = "congés de deuil"
contexte, sources = rechercher_contexte(test_question)
print(f"Test recherche : {len(sources)} sources trouvées")
print(f"Sources : {set(sources)}")

Test recherche : 3 sources trouvées
Sources : {'Convention collective - services alimentaires.pdf'}


## **Téléchargement Du Modèle `Llama 3.2 3B (GGUF)`**

Le téléchargement se fait directement dans le terminal pour éviter les problèmes de chemins avec les espaces et parenthèses.

**À exécuter dans le terminal :**

```bash
cd ../../models/1-rag/
wget https://huggingface.co/bartowski/Llama-3.2-3B-Instruct-GGUF/resolve/main/Llama-3.2-3B-Instruct-Q4_K_M.gguf

In [15]:
# Vérification du téléchargement
model_path = MODELS_DIR / "Llama-3.2-3B-Instruct-Q4_K_M.gguf"

if model_path.exists():
  print(f"Modèle trouvé : {model_path}")
  print(f"Taille : {model_path.stat().st_size / 1e9:.2f} GB")
else:
  print("Modèle non trouvé.")

Modèle trouvé : /home/kevin/Documents/Collège LaCité/Cours IA en Informatique (51847)/Hiver 2026/Intelligence Artificielle Générative (IFM31124-0020-H2026)/Projets/ua1/assistant-rag/models/1-rag/Llama-3.2-3B-Instruct-Q4_K_M.gguf
Taille : 2.02 GB


## **Chargement Du Modèle**

In [16]:
# Chargement de Llama 3.2 3B avec llama-cpp-python
from llama_cpp import Llama

print("Chargement du modèle en cours...")
llm = Llama(
  model_path=str(model_path),
  n_ctx=4096,
  n_threads=8,
  n_gpu_layers=0,
  verbose=False
)
print("Modèle chargé avec succès.")

Chargement du modèle en cours...


llama_context: n_ctx_per_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Modèle chargé avec succès.


## **Test De Génération Simple**

In [17]:
# Test rapide de génération
test_prompt = "Explique en une phrase ce qu'est un embedding vectoriel."

print("Génération en cours...")
output = llm(test_prompt, max_tokens=100, temperature=0.1)
reponse = output["choices"][0]["text"].strip()

print(f"Prompt : {test_prompt}")
print(f"Réponse : {reponse}")

Génération en cours...
Prompt : Explique en une phrase ce qu'est un embedding vectoriel.
Réponse : Un embedding vectoriel est une représentation vectorielle d'un élément de données, telle qu'un mot ou un image, qui capture ses caractéristiques et ses relations avec d'autres éléments de données de manière compacte et efficace.

Réponse : Un embedding vectoriel est une représentation vectorielle d'un élément de données qui capture ses caractéristiques et ses relations avec d'autres éléments de données de manière compacte et efficace. 

Cette réponse


## **Fonction De Génération `RAG` Complète**

In [25]:
# Fonction de génération avec retrieval intégré
def rag_repondre(question, k=3, max_tokens=500):
  """
  Pipeline RAG complet : retrieval + génération.
  """
  # 1. Retrieval
  contexte, sources = rechercher_contexte(question, k=k)
  
  # 2. Construction du prompt
  prompt = f"""En utilisant uniquement les extraits de documents fournis ci-dessous, réponds à la question de l'employé.

Extraits de la documentation :
{contexte}

Question de l'employé : {question}

Réponse (base-toi strictement sur les extraits fournis) :"""
  
  # 3. Génération
  output = llm(prompt, max_tokens=max_tokens, temperature=0.1)
  reponse = output["choices"][0]["text"].strip()
  
  return {
    "question": question,
    "sources": list(set(sources)),
    "reponse": reponse
  }

## **Test `RAG Complet`**

In [26]:
question_test = "Quels sont les congés de deuil prévus dans la convention collective?"

print(f"Question : {question_test}")
print("-" * 50)
resultat = rag_repondre(question_test)

print(f"Sources : {resultat['sources']}")
print(f"\nRéponse générée :\n{resultat['reponse']}")

Question : Quels sont les congés de deuil prévus dans la convention collective?
--------------------------------------------------
Sources : ['Convention collective - services alimentaires.pdf']

Réponse générée :
Selon l'Extrait 1, les employés ont droit à cinq (5) jours rémunérés en cas de décès de certains membres de la famille, notamment le père, la mère, l'époux(se), le conjoint de fait, l'enfant, le petit-enfant, le frère ou la sœur. Selon l'Extrait 2, pour le décès du grand-père, l'employé a droit à cinq (5) jours rémunérés. Selon l'Extrait 3, l'ancienneté est un critère important utilisé pour déterminer les éléments de la relation Employeur-employé, mais il n'y a pas de mention spécifique des congés de deuil. Cependant, l'Extrait 1 précise que les congés de deuil rémunérés ne sont pas accordés s'ils coïncident avec d'autres congés spécifiés dans la convention. Enfin, l'Extrait 2 précise que les congés de deuil sont octroyés pour le décès du père, de la mère, de l'époux(se), du 

## **Tests sur 12 Scénarios (Logs de Conversation)**

In [27]:
# Chargement des questions test
df_questions = pd.DataFrame([
  {"categorie": "Ressources humaines", "question": "Quels sont les congés de deuil prévus dans la convention collective?"},
  {"categorie": "Ressources humaines", "question": "Combien de jours de préavis pour une démission?"},
  {"categorie": "Entretien ménager", "question": "Quels sont les critères pour un hôtel 4 diamants concernant le service d'entretien ménager?"},
  {"categorie": "Entretien ménager", "question": "Comment un préposé doit-il répondre au téléphone selon les critères 4 diamants?"},
  {"categorie": "Réservations", "question": "Quelle est la procédure de gestion des réservations?"},
  {"categorie": "Réservations", "question": "Comment gérer une surréservation?"},
  {"categorie": "Politiques internes", "question": "Quelle est la tenue vestimentaire requise pour les employés?"},
  {"categorie": "Politiques internes", "question": "Quels sont les principes de vie internes de l'hôtel?"},
  {"categorie": "Santé et sécurité", "question": "Qui sont les membres du comité SST?"},
  {"categorie": "Santé et sécurité", "question": "Que faire en cas d'accident de travail?"},
  {"categorie": "Services alimentaires", "question": "Quelles sont les conditions de travail prévues par la convention collective pour les services alimentaires?"},
  {"categorie": "Services alimentaires", "question": "Comment fonctionne le comité patronal-syndical?"}
])

print(f"Nombre de scénarios : {len(df_questions)}")
df_questions.head()

Nombre de scénarios : 12


,categorie,question
0,Ressources humaines,Quels sont les congés de deuil prévus dans la ...
1,Ressources humaines,Combien de jours de préavis pour une démission?
2,Entretien ménager,Quels sont les critères pour un hôtel 4 diaman...
3,Entretien ménager,Comment un préposé doit-il répondre au télépho...
4,Réservations,Quelle est la procédure de gestion des réserva...


In [28]:
# Test sur tous les scénarios
resultats = []

for idx, row in df_questions.iterrows():
  print(f"\n=== Test {idx+1}/{len(df_questions)} : {row['categorie']} ===")
  print(f"Question : {row['question']}")
  
  # Exécution du RAG
  resultat = rag_repondre(row['question'])
  
  # Sauvegarde
  resultats.append({
    "categorie": row['categorie'],
    "question": row['question'],
    "sources": resultat['sources'],
    "reponse": resultat['reponse']
  })
  
  print(f"Sources : {resultat['sources']}")
  print(f"Réponse : {resultat['reponse'][:200]}...")
  print("-" * 50)

# Sauvegarde des résultats
import json
with open(MODELS_DIR / "resultats_rag_complets.json", "w") as f:
  json.dump(resultats, f, indent=2, ensure_ascii=False)
  
print("\nRésultats sauvegardés dans models/1-rag/resultats_rag_complets.json")


=== Test 1/12 : Ressources humaines ===
Question : Quels sont les congés de deuil prévus dans la convention collective?
Sources : ['Convention collective - services alimentaires.pdf']
Réponse : Selon l'Extrait 1, les employés ont droit à cinq (5) jours rémunérés en cas de décès de certains membres de la famille, notamment le père, la mère, l'époux(se), le conjoint de fait, l'enfant, le petit...
--------------------------------------------------

=== Test 2/12 : Ressources humaines ===
Question : Combien de jours de préavis pour une démission?
Sources : ['Convention collective - services alimentaires.pdf']
Réponse : Il n'y a pas d'indication dans les extraits fournis sur le préavis pour une démission. Les extraits traitent des cas de décès, de maladie, de blessures, d'absence pour cause de férié, etc. Il n'y a pa...
--------------------------------------------------

=== Test 3/12 : Entretien ménager ===
Question : Quels sont les critères pour un hôtel 4 diamants concernant le service 